In [ ]:
import numpy as np
import matplotlib.pyplot as plt
N = int(200)  # number of steps per loop of polishing

### Parameters:

In [ ]:
amp = 5  # parameter controlling the loop size in mm. The loop occupies a amp^2 mm^2 area
reps = 100  # number of polishing loops per site
x_step = 10  # x distance between sites' central locations in mm
y_step = 10  # y distance between sites' central locations in mm
z_step = 5  # height raised in moving sample between sites in mm

# define the desired number of sites in each direction
N_x_sites = 10
N_y_sites = 1

filename = 'AAAA'

# --- optional crystal slant compensation ---
use_slant = False  # set True to vary Z with position over a tilted surface
slant_x = 0.0  # mm height change per mm in X (dz/dx)
slant_y = 0.0  # mm height change per mm in Y (dz/dy)
z_ref = 0.0  # surface height at the origin (mm)

# alternatively, define tilt by angles (degrees) instead of slopes:
use_angles = False  # if True, slant_x/slant_y are computed from pitch/roll below
pitch_deg = 0.0  # tilt about X axis: height varies with Y
roll_deg = 0.0  # tilt about Y axis: height varies with X

if use_angles:
    slant_x = np.tan(np.deg2rad(roll_deg))
    slant_y = np.tan(np.deg2rad(pitch_deg))

# optional: fit slant from measured (x, y, z) probe points on the crystal
use_plane_fit = False
fit_x = np.array([0.0, 10.0, 0.0])
fit_y = np.array([0.0, 0.0, 10.0])
fit_z = np.array([0.0, 0.2, 0.1])  # example heights in mm

if use_plane_fit:
    A = np.column_stack([fit_x, fit_y, np.ones_like(fit_x)])
    slant_x, slant_y, z_ref = np.linalg.lstsq(A, fit_z, rcond=None)[0]
    use_slant = True

plot_path = False  # set True to preview one polishing loop in 3D

### Path Components:

In [ ]:
def surface_z(x, y, slant_x=0.0, slant_y=0.0, z_ref=0.0):
    """Height of a tilted crystal surface at (x, y)."""
    return z_ref + slant_x * x + slant_y * y


def make_polish_loop(amp, N, reps, use_slant=False, slant_x=0.0, slant_y=0.0, z_ref=0.0):
    """
    Build the figure-8 polishing path at a single site.
    Parametric form of y^4 = y^2 - x^2; Z follows the crystal surface when use_slant is True.
    """
    loop = []
    for theta in np.linspace(0, 2 * np.pi, N + 1):
        x = round(amp * 2 * np.sin(theta) * np.cos(theta), 4)
        y = round(amp * np.cos(theta - np.pi / 2), 4)
        if use_slant:
            z = round(surface_z(x, y, slant_x, slant_y, z_ref), 4)
        else:
            z = 0.0
        loop.append(np.array([x, y, z]))
    return loop * reps


def make_transit_paths(x_step, y_step, z_step, signed_x_step=None, use_slant=False, slant_x=0.0, slant_y=0.0):
    """
    Transit waypoints relative to the current site origin.
    signed_x_step: actual X displacement for a horizontal move (handles serpentine direction).
    """
    if signed_x_step is None:
        signed_x_step = x_step

    z_right = slant_x * signed_x_step if use_slant else 0.0
    z_down = -slant_y * y_step if use_slant else 0.0

    right_path = [
        np.array([0, 0, z_step]),
        np.array([signed_x_step, 0, z_step]),
        np.array([signed_x_step, 0, z_right]),
    ]
    down_path = [
        np.array([0, 0, z_step]),
        np.array([0, -y_step, z_step]),
        np.array([0, -y_step, z_down]),
    ]
    return right_path, down_path


# construct path for polishing at a single site
path_2d = make_polish_loop(amp, N, reps, use_slant, slant_x, slant_y, z_ref)

# default flat transit paths (used when use_slant is False)
right_path, down_path = make_transit_paths(x_step, y_step, z_step, use_slant=False)

In [ ]:
def generate_gcode_3d_from_path(path, feedrate=1000, z_safe=10):
    """
    Generates G-code from a list of 3D points.

    INPUTS:
        *   path: arr, list of tuples [(x1, y1, z1), (x2, y2, z2), ...] representing the 3D path.
        *   feedrate: int, the feed rate (speed of the tool) in units per minute.
        *   z_safe: float, safe Z height to move to when not cutting.

    OUTPUT:
        *   output: str, string containing the G-code.
    """
    gcode = []

    # Initialize machine
    gcode.append("G21")  # Set units to millimeters
    gcode.append("G90")  # Absolute positioning
    gcode.append(f"G0 Z{z_safe}")  # Move to safe height

    # Move to the start of the path without cutting
    start_x, start_y, start_z = path[0]
    gcode.append(f"G0 X{start_x} Y{start_y} Z{z_safe}")  # Move to start position
    gcode.append(f"G1 Z{start_z} F{feedrate}")  # Start cutting to the first Z height

    # Move along the path
    for (x, y, z) in path:
        gcode.append(f"G1 X{x} Y{y} Z{z}")  # Move to next point

    # Finish cutting by moving back to safe height
    gcode.append(f"G0 Z{z_safe}")  # Move back to safe height
    gcode.append("M30")  # End program

    return "\n".join(gcode)


def construct_full_path(
    path_2d,
    right_path,
    down_path,
    N_x_sites,
    N_y_sites,
    use_slant=False,
    slant_x=0.0,
    slant_y=0.0,
    x_step=0.0,
    y_step=0.0,
    z_step=0.0,
):
    """
    Constructs a full list of coordinates for the miller path.

    When use_slant is True, transit paths are rebuilt per row with the correct
    signed X direction and landing Z for the tilted surface.
    """
    total_path = []
    location = np.array([0.0, 0.0, 0.0])

    def append_right_transit(location, row_index):
        if use_slant:
            signed_x = x_step * ((-1) ** row_index)
            transit, _ = make_transit_paths(
                x_step, y_step, z_step, signed_x, use_slant=True, slant_x=slant_x, slant_y=slant_y
            )
            return [location + step for step in transit]
        return [location + np.array([(-1) ** row_index, 0, 1]) * step for step in right_path]

    def append_down_transit(location):
        if use_slant:
            _, transit = make_transit_paths(
                x_step, y_step, z_step, use_slant=True, slant_x=slant_x, slant_y=slant_y
            )
            return [location + step for step in transit]
        return [location + step for step in down_path]

    for j in range(N_y_sites - 1):
        total_path += [location + path_step for path_step in path_2d]
        location = total_path[-1]

        for _ in range(N_x_sites - 1):
            total_path += append_right_transit(location, j)
            location = total_path[-1]
            total_path += [location + path_step for path_step in path_2d]
            location = total_path[-1]

        total_path += append_down_transit(location)
        location = total_path[-1]

    total_path += [location + path_step for path_step in path_2d]
    location = total_path[-1]

    for _ in range(N_x_sites - 1):
        total_path += append_right_transit(location, N_y_sites - 1)
        location = total_path[-1]
        total_path += [location + path_step for path_step in path_2d]
        location = total_path[-1]

    return total_path

In [ ]:
# construct the full list of points to define the full path
total_path = construct_full_path(
    path_2d,
    right_path,
    down_path,
    N_x_sites,
    N_y_sites,
    use_slant=use_slant,
    slant_x=slant_x,
    slant_y=slant_y,
    x_step=x_step,
    y_step=y_step,
    z_step=z_step,
)

# convert the total_path into G-code
g_code = generate_gcode_3d_from_path(total_path, 1000000, 0)

In [ ]:
# write G-code to .txt file ready for input into the miller software
with open(filename, 'w') as file:
    file.write(g_code)

In [ ]:
print(filename)
if use_slant:
    print(f"Slant enabled: slant_x={slant_x:.6f} mm/mm, slant_y={slant_y:.6f} mm/mm, z_ref={z_ref:.4f} mm")

In [ ]:
# optional: preview a single figure-8 loop in 3D
if plot_path:
    single_loop = np.array(path_2d[: N + 1])
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot(single_loop[:, 0], single_loop[:, 1], single_loop[:, 2])
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    ax.set_zlabel('Z (mm)')
    ax.set_title('Single polishing loop')
    plt.tight_layout()
    plt.show()